In [ ]:
!git clone https://github.com/FlagOpen/FlagEmbedding.git

Cloning into 'FlagEmbedding'...
remote: Enumerating objects: 11727, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 11727 (delta 11), reused 10 (delta 10), pack-reused 11712 (from 2)
Receiving objects: 100% (11727/11727), 52.21 MiB | 12.08 MiB/s, done.
Resolving deltas: 100% (6501/6501), done.
Updating files: 100% (1152/1152), done.


In [ ]:
%cd FlagEmbedding

/content/drive/MyDrive/colab/MEmbER/FlagEmbedding


In [ ]:
!pip install  .

Processing /content/drive/MyDrive/colab/MEmbER/FlagEmbedding
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.2 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=flagembedding-1.4.2-py3-none-any.whl size=250528 sha256=5007ca82a12b90cb4d8165c28c5fa15f113a993b173977332689ef44f230b7eb
  Stored in directory: /tmp/pip-ephem-wheel-cache-_503c4lj/wheels/d7/f2/56/29113161e639cc82111b9ae78d0d72543e6b5f0f38507d9127
Successfully built FlagEmbedding


In [ ]:
!pip install nltk
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 56.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
import gensim
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
import transformers
from transformers import LlamaTokenizer, LlamaForCausalLM
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
import torch
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from transformers import BitsAndBytesConfig
import concurrent.futures
from math import log
from sklearn.decomposition import PCA
import torch.nn.functional as F
from FlagEmbedding import FlagAutoModel

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
def text_clear(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'\([^)]*\)', '', text)

    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    text = text.lower()

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
df = pd.read_csv('Irish_populated_places.csv')

In [ ]:
df["population"] = df["population"].fillna(0)
df = df.replace("nan", np.nan)
df = df.replace("NaN", np.nan)
df = df.replace("None", np.nan)
df["placeType"] = df["placeType"].fillna("")
df["broaderPlaceType"] = df["broaderPlaceType"].fillna("")

In [ ]:
df['subject'] = df['subject'].str.replace(
    'http://dbpedia.org/resource/Category:', '', regex=False
).map(lambda x: x.replace('_',' '))

In [ ]:
df['broaderSubject'] = df['broaderSubject'].fillna('').str.replace(
    'http://dbpedia.org/resource/Category:', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['country'] = df['country'].fillna('').str.replace(
    'http://dbpedia.org/resource/', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['placeType'] = df['placeType'].fillna('').str.replace(
    'http://dbpedia.org/resource/', '', regex=False
).str.replace('_', ' ')

In [ ]:
df['broaderPlaceType'] = df['broaderPlaceType'].fillna('').str.replace(
    'http://dbpedia.org/ontology/', '', regex=False
)

In [ ]:
df["population"] = df["population"].fillna(0)

In [ ]:
columns_to_clean = ['placeName', 'abstract', 'country', 'subject', 'broaderSubject', 'placeType', 'broaderPlaceType']
for col in columns_to_clean:
    df[col] = df[col].apply(text_clear)

In [ ]:
agg_df = df.groupby("place", as_index=False).agg({
    "placeName": "first",
    "abstract": "first",
    "country": "first",
    "subject": lambda x: "; ".join(set(x.dropna().astype(str))),
    "broaderSubject": lambda x: "; ".join(set(x.dropna().astype(str))),
    "placeType": lambda x: "; ".join(set(x.dropna().astype(str))),
    "broaderPlaceType": lambda x: "; ".join(set(x.dropna().astype(str))),
    "population": "first"
})

In [ ]:
agg_df['text'] = agg_df['placeName'] + " abstract:" + agg_df['abstract'] + " country:" + agg_df['country'] + " subject:" + agg_df['subject'] + " broaderSubject:" + agg_df['broaderSubject'] + " placeType:" + agg_df['placeType'] + " broaderPlaceType:" + agg_df['broaderPlaceType'] + " population:" + agg_df['population'].astype(str)

In [ ]:
places = agg_df['place'].tolist()
texts = agg_df['text'].tolist()

In [ ]:
model = FlagAutoModel.from_finetuned('BAAI/bge-base-en-v1.5',
                                      query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
                                      use_fp16=True)

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
embeddings_array = model.encode(texts, batch_size=256)
embeddings_tensor = torch.tensor(embeddings_array, dtype=torch.float32)

Inference Embeddings: 100%|██████████| 2/2 [09:58<00:00, 299.44s/it]


In [ ]:
text_emb_dict = {
    place: embeddings_tensor[i]
    for i, place in enumerate(places)
}

In [ ]:
def compute_similar_places(entity_repr, top_k=10):
    entities = list(entity_repr.keys())
    all_embeddings = torch.stack([entity_repr[e] for e in entities])  # [N, D]
    sims = F.cosine_similarity(all_embeddings.unsqueeze(1), all_embeddings.unsqueeze(0), dim=-1)

    similar_places_dict = {}
    for i, e in enumerate(entities):
        sim_scores = sims[i]
        topk = torch.topk(sim_scores, k=top_k+1)  # +1 to skip itself
        indices = topk.indices.tolist()
        similar_entities = [entities[j] for j in indices if j != i][:top_k]
        similar_places_dict[e] = similar_entities
    return similar_places_dict

In [ ]:
similar_places = compute_similar_places(text_emb_dict, top_k=10)

In [ ]:
ground_truth = pd.read_csv("./ground_truth.csv")

In [ ]:
ground_truth_filtered = (
    ground_truth
    .sort_values(
        by=["geographical_entity_1", "F-score"],
        ascending=[True, False]
    )
    .groupby("geographical_entity_1")
    .head(20)
)
y_true_dict = ground_truth_filtered.groupby('geographical_entity_1')['geographical_entity_2'].apply(list).to_dict()

def evluation(k, y_true_dict, similar_places_dict):
  invalid_places = []
  # Compute metrics
  precisions, recalls, ndcgs, hits, map_scores,mrrs = [], [], [], [], [], []
  place_idxs = list(y_true_dict.keys())

  for pid in place_idxs:
      if pid not in similar_places_dict or len(similar_places_dict[pid]) < k:
          invalid_places.append(pid)
          continue
      pred_list, rel_set = similar_places_dict[pid][:k], y_true_dict[pid]
      pred_real = "pid:"+str(pid)+' '+"pred_list:"+str(pred_list)+' '+"rel_set:"+str(rel_set)

      if len(pred_list) == 0:
          continue


      dcg = 0.0
      hit_num = 0.0
      for i in range(len(pred_list)):
          if pred_list[i] in rel_set:
              dcg += 1. / (log(i + 2) / log(2))
              hit_num += 1
      # idcg
      idcg = 0.0
      for i in range(min(len(rel_set), len(pred_list))):
          idcg += 1. / (log(i + 2) / log(2))
      ndcg = dcg / idcg
      recall = hit_num / len(rel_set)
      precision = hit_num / len(pred_list)
      hit = 1.0 if hit_num > 0.0 else 0.0

      #map
      map_score = 0.0
      num_hits = 0.0
      score = 0.0
      for i,p in enumerate(pred_list):
          if p in rel_set and p not in pred_list[:i]:
              num_hits+=1.0
              score+=num_hits/(i+1.0)
      map_score = score/min(len(rel_set),k)
      #map_score = score / min(len(rel_set), len(pred_list)) if len(rel_set) > 0 else 0.0

      #MRR
      rr = 0.0
      for i, p in enumerate(pred_list):
          if p in rel_set:
              rr = 1.0 / (i + 1.0)
              break

      ndcgs.append(ndcg)
      recalls.append(recall)
      precisions.append(precision)
      hits.append(hit)
      map_scores.append(map_score)
      mrrs.append(rr)

  avg_precision = np.mean(precisions) * 100
  avg_recall = np.mean(recalls) * 100
  avg_ndcg = np.mean(ndcgs) * 100
  avg_hit = np.mean(hits) * 100
  avg_map = np.mean(map_scores) * 100
  avg_mrr = np.mean(mrrs) * 100

  print("invalid places:", str(len(invalid_places)))
  print('MAP={:.3f} | NDCG={:.3f} |  Recall={:.3f} | Precision={:.3f} | Hits={:.3f} | MRR={:.3f}'.format(
          avg_map, avg_ndcg, avg_recall, avg_precision, avg_hit, avg_mrr))

evluation(k=10, y_true_dict=y_true_dict, similar_places_dict=similar_places)


invalid places: 0
MAP=23.268 | NDCG=36.996 |  Recall=17.254 | Precision=33.925 | Hits=92.491 | MRR=62.535
